<a id="top"></a>

# L12 lecture: Build the eval harness — case, scorer, runner

```yaml
title: "L12 lecture: Build the eval harness — case, scorer, runner"
keywords: evaluation, eval case, scorer, runner, outcome scoring, trajectory scoring, regression case, brittleness, teacher lecture
estimated duration: 35
```

> **Lesson:** L12 Evaluation. Roadmap: [objectives.md](../../../../docs/origin/lesson_roadmaps/L12/objectives.md) (objectives 1 & 2).
>
> Runs offline — no API key needed (scripted `FakeModel` from L11).

## Contents

- [1. Setup — runs to score](#1-setup--runs-to-score)
- [2. The eval harness, made visible](#2-the-eval-harness-made-visible)
  - [2.1 A case is an input plus what "good" means](#21-a-case-is-an-input-plus-what-good-means)
  - [2.2 A scorer turns one run into a verdict](#22-a-scorer-turns-one-run-into-a-verdict)
  - [2.3 The runner runs every case and reports](#23-the-runner-runs-every-case-and-reports)
  - [2.4 Score the path, not only the answer](#24-score-the-path-not-only-the-answer)
- [3. From a trace to a regression case](#3-from-a-trace-to-a-regression-case)
  - [3.1 The runaway becomes a trajectory check](#31-the-runaway-becomes-a-trajectory-check)
  - [3.2 The tool-error becomes an outcome check](#32-the-tool-error-becomes-an-outcome-check)
  - [3.3 The brittleness beat](#33-the-brittleness-beat)
- [4. Takeaways](#4-takeaways)

## 1. Setup — runs to score

L11 produced the record (the trace); L12 **judges it**. To judge runs we first need runs. We drive the **same** `agent_loop.run(...)` from L10/L11 with a scripted `FakeModel`, so every run is deterministic and keyless — the focus stays on *eval design*, not on coaxing a live model.

The new import is `fluffy_potato_curriculum.common.evals` — the eval harness this lesson is about. It is ~30 lines of plain Python you can read end-to-end; open `common/evals.py` alongside this notebook.

A **`run_case`** is the thing under test: given a case, it produces one `RunResult`. Ours looks the case up in a script table and runs the loop. (Two `run_case`s over the same cases — one per model — is the A/B in the next lecture.)

In [1]:
from langchain_core.messages import AIMessage

from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    evaluate,
    tool_calls,
    tool_trajectory,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS

TASK_CHAIN = "What is 17 * 23, and what is the population of Tokyo?"

# A script table: case id -> the model's scripted moves for that case. FakeModel
# repeats its LAST line, which is how a one-line script becomes a runaway.
SCRIPTS: dict[str, list[AIMessage]] = {
    # clean two-tool run; the answer contains "37000000" (no commas) on purpose.
    "tokyo_pop": [
        tool_reply(tool_call("c1", "calculator", {"expression": "17*23"})),
        tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
        text_reply("17 * 23 is 391, and Tokyo has a population of 37000000."),
    ],
}


def run_case(case: EvalCase) -> RunResult:
    """Run the loop for one case using its scripted FakeModel (a fresh one each call).

    ``max_steps`` is read from the case inputs so a runaway case can cap low.
    """
    model = FakeModel(SCRIPTS[case.id])
    return run(model, TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8))


# One case, carrying BOTH an expected answer and an expected trajectory.
tokyo_case = EvalCase(
    id="tokyo_pop",
    inputs={"task": TASK_CHAIN},
    reference_outputs={"answer": "37000000", "expected_tools": ["calculator", "lookup"]},
)

demo_run = run_case(tokyo_case)
print("termination:", demo_run.termination, "| iterations:", demo_run.iterations)
print("final_text:", demo_run.final_text)

termination: natural | iterations: 3
final_text: 17 * 23 is 391, and Tokyo has a population of 37000000.


[↑ Back to top](#top)

## 2. The eval harness, made visible

Three nouns and one verb, and you can read every one of them:

- a **case** (`EvalCase`) — an input (`inputs`) plus what "good" means (`reference_outputs`);
- a **scorer** (`Scorer`) — a function `(*, run, example) -> EvalResult` that turns one run into a verdict;
- a **runner** (`evaluate`) — runs every case and reports a summary.

The whole "framework" is a function from `(run, example)` to a verdict, called in a loop. Nothing magic.

### 2.1 A case is an input plus what "good" means

`EvalCase` is a typed record — like an L11 trace event. `inputs` seeds the run; `reference_outputs` is the answer key a scorer checks against. These field names line up with LangSmith's *Example* (`inputs` / `reference_outputs`), so they transfer if you adopt a platform later.

In [2]:
print("id:               ", tokyo_case.id)
print("inputs:           ", tokyo_case.inputs)
print("reference_outputs:", tokyo_case.reference_outputs)

id:                tokyo_pop
inputs:            {'task': 'What is 17 * 23, and what is the population of Tokyo?'}
reference_outputs: {'answer': '37000000', 'expected_tools': ['calculator', 'lookup']}


### 2.2 A scorer turns one run into a verdict

A scorer reads `run.final_text` for **outcome** scoring (did it get the right answer?). It returns an `EvalResult` — a `key` (the metric name), a `score` (pass/fail), and an optional `comment`. Here is the first scorer, `answer_correct`: does the final text contain the reference answer?

In [3]:
def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer: did the final text contain the reference answer (a substring)?"""
    expected = str(example.reference_outputs["answer"])
    return EvalResult(
        key="answer_correct",
        score=expected in run.final_text,
        comment=f"looking for {expected!r}",
    )


verdict = answer_correct(run=demo_run, example=tokyo_case)
print(verdict)

key='answer_correct' score=True comment="looking for '37000000'"


### 2.3 The runner runs every case and reports

`evaluate(run_case, cases, scorers)` is the loop: for each case it calls `run_case(case)` to produce a run, applies every scorer, and rolls the verdicts up into a table. Each cell reads `passes/samples` — here `samples=1`, so it's `1/1` (pass) or `0/1` (fail). The `ALL` column is the samples where *every* scorer passed.

In [4]:
report = evaluate(run_case, [tokyo_case], [answer_correct])
print(report.table())

case       answer_correct  ALL
tokyo_pop  1/1             1/1


### 2.4 Score the path, not only the answer

For an agent, *how* it got there matters: a right answer reached by a runaway, over-budget path is still a problem. A **trajectory** scorer reads `run.trace` (via `tool_trajectory`, which lists the ordered tool-call names) instead of `run.final_text`. That trace is exactly the L11 artifact — which is *why* L11 came first.

Add `expected_tools` and re-run. Now the **same run is scored two ways**: did it get the right answer, *and* did it take the right path?

In [5]:
def expected_tools(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: did the ordered tool calls match the reference path?"""
    expected = list(example.reference_outputs["expected_tools"])
    actual = tool_trajectory(run)
    return EvalResult(
        key="expected_tools",
        score=actual == expected,
        comment=f"{actual} vs expected {expected}",
    )


report = evaluate(run_case, [tokyo_case], [answer_correct, expected_tools])
print(report.table())
print()
print("trajectory was:", tool_trajectory(demo_run))

case       answer_correct  expected_tools  ALL
tokyo_pop  1/1             1/1             1/1

trajectory was: ['calculator', 'lookup']


An eval set can score the **answer**, the **path**, or **both**. Pick deliberately — for agents the path often matters as much as the answer.

[↑ Back to top](#top)

## 3. From a trace to a regression case

The most important rule in this lesson:

> **trace a failure → write a case that catches it → keep the case forever.**

Good eval cases come from *observed* failures, not imagination. L11 handed you four failure signatures — tool error, wrong arguments, runaway loop, premature termination. Each becomes a **regression case**: a check that **fails when the bug is present and passes when it's fixed**. A "case" that passes no matter what tests nothing.

### 3.1 The runaway becomes a trajectory check

Recall the L11 runaway signature in one line — *the same `(tool_name, args)` pair repeating, ending in `max_steps`*. We don't re-teach reading it; we turn it into a scorer. `no_runaway` reads `tool_calls` (names **with** args) and the termination cause, and fails if either red flag is present.

We add two cases on the same Atlantis task: the broken one (runs away) and the fixed one (gives up cleanly). The check must fail on the first and pass on the second.

In [6]:
SCRIPTS["atlantis_runaway"] = [
    # one line; FakeModel repeats it, so lookup('Atlantis') fires until max_steps.
    tool_reply(tool_call("c1", "lookup", {"city": "Atlantis"})),
]
SCRIPTS["atlantis_fixed"] = [
    # the "fixed" agent gives up cleanly with text only -> natural termination.
    text_reply("I do not have a population on file for Atlantis."),
]


def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: no repeated (tool, args) call, and didn't hit the step cap."""
    calls = tool_calls(run)
    signatures = [(name, tuple(sorted(args.items()))) for name, args in calls]
    repeated = len(signatures) != len(set(signatures))
    hit_cap = run.termination == "max_steps"
    return EvalResult(
        key="no_runaway",
        score=not (repeated or hit_cap),
        comment=f"termination={run.termination}, tool_calls={len(calls)}",
    )


runaway_case = EvalCase(
    id="atlantis_runaway", inputs={"task": "Population of Atlantis?", "max_steps": 4}
)
fixed_case = EvalCase(id="atlantis_fixed", inputs={"task": "Population of Atlantis?"})

report = evaluate(run_case, [runaway_case, fixed_case], [no_runaway])
print(report.table())

case              no_runaway  ALL
atlantis_runaway  0/1         0/1
atlantis_fixed    1/1         1/1


`atlantis_runaway` scores `0/1` (the bug is present); `atlantis_fixed` scores `1/1` (the bug is gone). That is the definition of a regression case: **red when broken, green when fixed.**

### 3.2 The tool-error becomes an outcome check

The `flaky_fetch` tool fails in known ways (L11's raw material). `https://ok` returns a usable value; `https://crash` raises, and the loop converts it to a `ToolMessage(status="error")`. A *good* run recovers the answer; a *broken* run gives up. An **outcome** check on the final text catches the difference — fails when the answer never recovered.

In [7]:
SCRIPTS["fetch_ok"] = [
    tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://ok"})),
    text_reply("The page says: the answer is 42."),
]
SCRIPTS["fetch_crash"] = [
    tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
    text_reply("Sorry, I could not fetch that URL."),  # gave up
]

fetch_ok_case = EvalCase(
    id="fetch_ok",
    inputs={"task": "Fetch https://ok and tell me the answer."},
    reference_outputs={"answer": "42"},
)
fetch_crash_case = EvalCase(
    id="fetch_crash",
    inputs={"task": "Fetch https://crash and tell me the answer."},
    reference_outputs={"answer": "42"},
)

report = evaluate(run_case, [fetch_ok_case, fetch_crash_case], [answer_correct])
print(report.table())

case         answer_correct  ALL
fetch_ok     1/1             1/1
fetch_crash  0/1             0/1


`fetch_ok` recovers the answer (`1/1`); `fetch_crash` gives up (`0/1`). Same scorer, two runs, one regression caught.

### 3.3 The brittleness beat

A check can be *too tight*. An exact/substring match on a number is fragile: a correct answer that's merely **reworded** trips it. A red that fires on a correct run is worse than no check — it trains you to ignore reds.

Watch `answer_correct` fail on a run that is *right* but writes `37,000,000` with commas.

In [8]:
SCRIPTS["tokyo_reworded"] = [
    tool_reply(tool_call("c1", "calculator", {"expression": "17*23"})),
    tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
    # correct, but written with commas -> the substring "37000000" is NOT present.
    text_reply("17 x 23 = 391, and about 37,000,000 people live in Tokyo."),
]
reworded_case = EvalCase(
    id="tokyo_reworded",
    inputs={"task": TASK_CHAIN},
    reference_outputs={"answer": "37000000"},
)

print("strict substring check:")
print(evaluate(run_case, [reworded_case], [answer_correct]).table())

strict substring check:
case            answer_correct  ALL
tokyo_reworded  0/1             0/1


It reads `0/1` even though the answer is right. The fix is the **loosest check that still catches the bug** — here, normalize away commas and spaces before comparing. (When even a normalized check can't express the quality — "did the answer *acknowledge the failure gracefully*?" — you reach for an LLM-as-judge. We name that now and build a minimal one in `L1206_lecture`.)

In [9]:
def _digits_only(text: str) -> str:
    """Drop commas and spaces so '37,000,000' and '37000000' compare equal."""
    return text.replace(",", "").replace(" ", "")


def answer_correct_loose(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer, loosened: compare with commas/spaces normalized out."""
    expected = str(example.reference_outputs["answer"])
    return EvalResult(
        key="answer_correct",
        score=_digits_only(expected) in _digits_only(run.final_text),
        comment="normalized digits",
    )


print("loosened check:")
print(evaluate(run_case, [reworded_case], [answer_correct_loose]).table())

loosened check:
case            answer_correct  ALL
tokyo_reworded  1/1             1/1


[↑ Back to top](#top)

## 4. Takeaways

- **Three nouns, one verb.** A **case** is an input plus what "good" means; a **scorer** turns one run into a verdict; the **runner** (`evaluate`) runs every case and reports. The harness is ~30 readable lines, not a framework.
- **Score the answer, the path, or both.** Outcome scorers read `final_text`; trajectory scorers read `run.trace`. For agents the path often matters as much as the answer — which is why L11 (tracing) came first.
- **Cases come from real failures.** **Trace a failure → write a case that catches it → keep it forever.** A regression case is red when broken and green when fixed; one that passes no matter what tests nothing.
- **Use the loosest check that still catches the bug.** An over-tight check (exact string) trains you to ignore reds; escalate to a semantic/LLM-judge only when a cheap check genuinely can't express the quality.
- **Next:** [`L1203_lab`](L1203_lab_empty.ipynb) — you write your own cases and scorers, including one regression case from an L11 failure. Then [`L1204_lecture`](L1204_lecture.ipynb) confronts non-determinism and moves from a verdict to a **pass rate**.

[↑ Back to top](#top)